# BIXI April 2019: live demo
Describe, prepare, profile, explore and visualize one month of real BIXI trips. This notebook follows the deck step by step.

**Files** (upload both to Colab: folder icon on the left, then the upload icon): `OD_2019-04.csv` (trips, as published by BIXI) and `Stations_2019.csv` (station list).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---
# Part 1: The BIXI dataset
What BIXI is, which files it publishes, what each column means, and what the data can and cannot tell us.

---
# Part 2: Prepare
Load the raw files, fix the types, remove duplicates, merge station names, and create the columns the questions need.

## Load the raw trip file
parse_dates converts the two date columns from text to real dates while reading the file, so that the hour, the day or the date can be extracted later with .dt.

**Class question:** what would happen if we forgot parse_dates and tried trips['start_date'].dt.hour?

In [ ]:
trips = pd.read_csv('OD_2019-04.csv', parse_dates=['start_date', 'end_date'])
print(trips.shape)
trips.dtypes

In [ ]:
trips.head()

## Duplicates
duplicated() marks rows that are exact copies of an earlier row (same start and end time to the second, same stations, same duration, same rider type). There are 10.

**Class question:** 10 duplicates out of 237,542 rows: is it worth the trouble? (Yes: the check costs one line, and in other files duplicates can be thousands of rows, as in the full 2019 data.)

In [ ]:
print('duplicated rows:', trips.duplicated().sum())
trips = trips.drop_duplicates().copy()
print(trips.shape)

## Merge the station names
The trip file has codes; people understand names. A left merge with the station list adds the names, once for the start station and once for the end station.

**Class question:** what would an inner join (how='inner') have done here, and why is it a worse choice?

In [ ]:
stations = pd.read_csv('Stations_2019.csv')
start = stations.rename(columns={'Code': 'start_station_code', 'name': 'start_name'})
end = stations.rename(columns={'Code': 'end_station_code', 'name': 'end_name'})
trips = trips.merge(start[['start_station_code', 'start_name']], how='left')
trips = trips.merge(end[['end_station_code', 'end_name']], how='left')
print(trips.shape)
print(trips[['start_name', 'end_name']].isna().sum())

## The unknown stations
Which codes did not match? Only two: 6708 and 6034, both used hundreds of times. They are real stations that are not in the 2019 station list, perhaps temporary stations, stations moved during the season, or stations added after the list was produced.

**Class question:** how could we find the location of these two stations? (Look them up in other years' station files, or use the coordinates of the stations their trips connect to.)

In [ ]:
print(trips.loc[trips['start_name'].isna(), 'start_station_code'].value_counts())
trips['start_name'] = trips['start_name'].fillna('unknown station')
trips['end_name'] = trips['end_name'].fillna('unknown station')

## New columns for the questions
The questions of Part 1 need columns that are not in the file. Each one comes from an existing column:

**Class question:** is hour a numerical or a categorical variable? (It is a number, but for counting trips per hour it is used like a category; for a scatter plot it is used as a number. The use decides.)

In [ ]:
trips['duration_min'] = trips['duration_sec'] / 60
trips['rider'] = trips['is_member'].map({1: 'member', 0: 'occasional'})
trips['hour'] = trips['start_date'].dt.hour
trips['day'] = trips['start_date'].dt.day_name()
trips['weekend'] = trips['start_date'].dt.dayofweek >= 5
trips['round_trip'] = trips['start_station_code'] == trips['end_station_code']
trips[['duration_min', 'rider', 'hour', 'day', 'weekend', 'round_trip']].head(3)

---
# Part 3: Profile
Check ranges, consistency and categories, and summarise the data in a profiling report.

## Ranges and consistency
Profiling checks that the values are plausible before any analysis.

**Class question:** what else would you check for plausibility? (End after start; stations in Montreal; no trip ending before the season opens; is_member only 0 or 1.)

In [ ]:
print(trips['duration_sec'].describe())
diff = (trips['end_date'] - trips['start_date']).dt.total_seconds() - trips['duration_sec']
print('largest difference (s):', diff.abs().max())
print(trips['start_date'].min(), trips['start_date'].max())

## Categories
value_counts on the categorical columns shows their values and their balance.

**Class question:** would you expect the member share to be the same in July? Why?

In [ ]:
print(trips['rider'].value_counts())
print(trips['rider'].value_counts(normalize=True).round(3))
print('start stations used:', trips['start_station_code'].nunique())

---
# Part 4: Explore
Answer the questions of Part 1 with descriptive statistics and simple group comparisons.

## How long are trips?
The mean trip lasts 12.9 minutes but the median only 9.9: the distribution is right-skewed (skewness 2.55), with many short trips and a long tail of leisure rides. The median is the better "typical" value.

**Class question:** BIXI includes a fixed number of minutes in each trip. Based on these numbers, would a shorter included time affect members or occasional riders more?

In [ ]:
print(trips['duration_min'].describe().round(2))
print('skewness:', round(trips['duration_min'].skew(), 2))
trips.groupby('rider')['duration_min'].median().round(2)

## Who rides, and how?
A profile of the two rider types with named aggregation (agg with new column names): number of trips, median duration, share of weekend trips and share of round trips.

**Class question:** how would you use this profile to design a campaign that converts occasional riders into members?

In [ ]:
trips.groupby('rider').agg(trips=('rider', 'size'),
                            median_min=('duration_min', 'median'),
                            weekend=('weekend', 'mean'),
                            round_trip=('round_trip', 'mean')).round(3)

## Which days? Beware of totals
A trap with partial periods. Counting trips per day of the week gives Tuesday and Monday far ahead, and Friday last. But April 14 to 30, 2019 contains 3 Sundays, 3 Mondays and 3 Tuesdays, and only 2 of each other day: totals favour the days that appear more often.

**Class question:** what other comparisons can be biased by unequal numbers of days (or of stations, or of riders)?

In [ ]:
totals = trips['day'].value_counts()
days_in_data = trips['start_date'].dt.normalize().drop_duplicates().dt.day_name().value_counts()
per_day = (totals / days_in_data).round(0).sort_values(ascending=False)
pd.DataFrame({'total': totals, 'days': days_in_data, 'per_day': per_day})

## Which hours?
Trips per hour, separately for weekdays and weekends. On weekdays the busiest hour is 17:00 and the second peak is in the morning around 8:00: the commuting pattern (to work in the morning, home in the evening). On weekends the busiest hour is 15:00: an afternoon, leisure pattern, with no morning peak.

**Class question:** when would be the best time for maintenance crews to work on weekday stations?

In [ ]:
by_hour = trips.groupby(['weekend', 'hour']).size().unstack(0)
by_hour.columns = ['weekday', 'weekend']
print(by_hour.idxmax())
by_hour.loc[[7, 8, 12, 15, 17]]

## Which stations?
The busiest start stations, with their names (from the merge of Part 2). The top of the list is dominated by stations near metro stations and in the Plateau and downtown: Métro Laurier and Métro Mont-Royal lead.

**Class question:** why might a station near a metro lose bikes, while a station downtown gains them?

In [ ]:
print(trips['start_name'].value_counts().head(5))
flow = pd.concat([trips['start_name'].value_counts(),
                  trips['end_name'].value_counts()], axis=1, keys=['departures', 'arrivals']).fillna(0)
flow['net'] = flow['arrivals'] - flow['departures']
flow.sort_values('net').iloc[[0, 1, -2, -1]]

---
# Part 5: Visualize
Four essential charts for this dataset. Lecture 11 covers chart choice, design and patterns in depth.

## Trip durations
A histogram answers "how long are trips?" at a glance: most trips last between 3 and 15 minutes, with a long tail to the right up to the 2-hour limit. The red line is the median.

**Class question:** how would this histogram look for occasional riders only?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.hist(trips['duration_min'], bins=range(0, 122, 2), color='teal')
ax.axvline(trips['duration_min'].median(), color='red', label='median')
ax.set_xlabel('Trip duration (minutes)'); ax.set_ylabel('Trips')
ax.set_title('Most trips last 3 to 15 minutes'); ax.legend()
plt.show()

## Trips by hour
The by_hour table of Part 4 drawn as two lines: weekdays show the two commuting peaks (8 a.m. and 5 p.m.); weekends show one broad afternoon hump.

**Class question:** what would happen to the weekday line on a holiday Monday?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(by_hour.index, by_hour['weekday'], label='weekdays')
ax.plot(by_hour.index, by_hour['weekend'], label='weekends')
ax.set_xlabel('Hour of the day'); ax.set_ylabel('Trips')
ax.set_title('Weekdays: two commuting peaks'); ax.legend()
plt.show()

## Trips per day in April
The daily total from the opening day to April 30. The number of trips varies a lot from day to day: from 4,673 on April 19 to 23,365 on April 30.

**Class question:** what data would you add to explain the big daily jumps?

In [ ]:
daily = trips.groupby(trips['start_date'].dt.date).size()
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(daily.index, daily.values, marker='o')
ax.set_ylabel('Trips per day')
ax.set_title('April 2019: large day-to-day swings')
fig.autofmt_xdate(); plt.show()

## The busiest stations
A horizontal bar chart of the 10 busiest start stations: long names fit on the y axis, and invert_yaxis puts the busiest at the top.

**Class question:** how would you show both the number of departures and the net flow on one chart?

In [ ]:
top = trips['start_name'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.barh(top.index, top.values, color='teal')
ax.invert_yaxis(); ax.set_xlabel('Departures in April 2019')
ax.set_title('Metro stations lead')
plt.show()

---
# Part 6: From findings to insights
Turning numbers into statements that support decisions.

---
## Your turn
Pick one question from the deck that we did not answer (for example, the 10 most frequent start-end routes, or the stations with the most round trips) and answer it with a number, a table or a chart.